# Pre-Annotate Prescription Sentences (Auto-Tagging Assistant)

Manually drawing every DRUG/DOSAGE/FREQUENCY/DURATION span across 700 sentences
by hand is extremely slow (~57 spans in a single dense polypharmacy sentence).

This notebook auto-tags your `annotation_batch_top700.csv` using:
- **scispaCy** (`en_ner_bc5cdr_md` model) → tags `CHEMICAL` (mapped to `DRUG`) and
  `DISEASE` (mapped to `DIAGNOSIS`)
- **Regex patterns** (same ones from the prioritization notebook) → tag
  `DOSAGE`, `FREQUENCY`, `DURATION`

It then exports a **Label Studio pre-annotation JSON** file. When you import
this into Label Studio (instead of your original CSV), every sentence already
has its spans drawn and tagged — you just review, fix mistakes, add any
missed `SYMPTOM` tags (not auto-tagged, since it's a fuzzier category), and
hit Submit. This should cut your annotation time dramatically.


## 1. Install dependencies

`scispacy` + a pretrained biomedical NER model (`en_ner_bc5cdr_md`, ~90MB).
This only needs to run once.

In [1]:
%pip install scispacy pandas -q
%pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import json
import pandas as pd
import spacy

INPUT_CSV = "../data/processed/annotation_batch_top700.csv"
OUTPUT_JSON = "../data/annotated/preannotated_tasks.json"

nlp = spacy.load("en_ner_bc5cdr_md")
print("scispaCy model loaded. Entity labels it can produce:", nlp.get_pipe("ner").labels)

C:\Users\rohit\AppData\Local\Programs\Python\Python312\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


scispaCy model loaded. Entity labels it can produce: ('CHEMICAL', 'DISEASE')


## 2. Regex patterns for DOSAGE / FREQUENCY / DURATION

Same logic as the prioritization notebook, but here we need the **character
spans** (start/end offsets), not just counts, since Label Studio needs exact
positions to draw the highlight.

In [3]:
DOSAGE_RE = re.compile(
    r"\b\d+(\.\d+)?\s?(mg|mcg|ml|g|units?|tablets?|tabs?|sprays?|drops?|puffs?|capsules?|mEq)\b",
    re.IGNORECASE,
)

FREQUENCY_RE = re.compile(
    r"\b(once|twice|three times|four times)\s+(a\s+)?(day|daily)\b"
    r"|\bevery\s+\d+\s+hours?\b"
    r"|\b(b\.i\.d\.?|t\.i\.d\.?|q\.i\.d\.?|q\.d\.?|p\.r\.n\.?|bid|tid|qid|qd|prn)\b"
    r"|\b\d-\d-\d\b"
    r"|\bdaily\b",
    re.IGNORECASE,
)

DURATION_RE = re.compile(
    r"\bfor\s+\d+\s+(day|days|week|weeks|month|months)\b"
    r"|\bfor\s+(a|one|two|three|four|five|six)\s+(day|days|week|weeks|month|months)\b"
    r"|\bx\d+d\b",
    re.IGNORECASE,
)

REGEX_LABEL_MAP = [
    (DOSAGE_RE, "DOSAGE"),
    (DURATION_RE, "DURATION"),   # check duration before frequency: "for 3 days" shouldn't get eaten by "daily"-style frequency matches
    (FREQUENCY_RE, "FREQUENCY"),
]

def regex_spans(sentence: str):
    """Return non-overlapping (start, end, label) spans from regex patterns."""
    spans = []
    claimed = [False] * len(sentence)
    for pattern, label in REGEX_LABEL_MAP:
        for m in pattern.finditer(sentence):
            s, e = m.start(), m.end()
            if any(claimed[s:e]):
                continue  # skip if this text already tagged by an earlier (higher priority) pattern
            spans.append((s, e, label))
            for i in range(s, e):
                claimed[i] = True
    return spans

## 3. scispaCy spans (DRUG / DIAGNOSIS)

`CHEMICAL` → `DRUG`, `DISEASE` → `DIAGNOSIS`. We skip any scispaCy span that
overlaps a span already claimed by regex (dosage/frequency/duration numbers
occasionally get mis-flagged as chemicals).

In [4]:
SCISPACY_LABEL_MAP = {
    "CHEMICAL": "DRUG",
    "DISEASE": "DIAGNOSIS",
}

def model_spans(sentence: str, already_claimed_spans):
    claimed = [False] * len(sentence)
    for s, e, _ in already_claimed_spans:
        for i in range(s, e):
            claimed[i] = True

    doc = nlp(sentence)
    spans = []
    for ent in doc.ents:
        mapped_label = SCISPACY_LABEL_MAP.get(ent.label_)
        if not mapped_label:
            continue
        s, e = ent.start_char, ent.end_char
        if any(claimed[s:e]):
            continue
        spans.append((s, e, mapped_label))
        for i in range(s, e):
            claimed[i] = True
    return spans

## 4. Run the full pipeline on one example sentence (sanity check)

In [5]:
example = ("Triglide 50 mg daily, Prevacid 30 mg daily, Plavix 75 mg daily, "
           "potassium 10 mEq daily, Lasix 60 mg daily.")

r_spans = regex_spans(example)
m_spans = model_spans(example, r_spans)
all_spans = sorted(r_spans + m_spans, key=lambda x: x[0])

for s, e, label in all_spans:
    print(f"{label:12s} -> {example[s:e]!r}")

DOSAGE       -> '50 mg'
FREQUENCY    -> 'daily'
DOSAGE       -> '30 mg'
FREQUENCY    -> 'daily'
DRUG         -> 'Plavix'
DOSAGE       -> '75 mg'
FREQUENCY    -> 'daily'
DRUG         -> 'potassium'
DOSAGE       -> '10 mEq'
FREQUENCY    -> 'daily'
DRUG         -> 'Lasix'
DOSAGE       -> '60 mg'
FREQUENCY    -> 'daily'


## 5. Build Label Studio prediction JSON for all 700 sentences

Label Studio's pre-annotation format: each task needs a `data` field (your
sentence) plus a `predictions` list containing `result` entries with
`from_name`, `to_name`, and the labeled span.

In [6]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} sentences to pre-annotate.")

tasks = []
skipped = 0

for _, row in df.iterrows():
    sentence = str(row["sentence"])
    try:
        r_spans = regex_spans(sentence)
        m_spans = model_spans(sentence, r_spans)
    except Exception as e:
        skipped += 1
        continue

    all_spans = r_spans + m_spans

    results = []
    for i, (s, e, label) in enumerate(all_spans):
        results.append({
            "id": f"span_{i}",
            "from_name": "label",
            "to_name": "text",
            "type": "labels",
            "value": {
                "start": s,
                "end": e,
                "text": sentence[s:e],
                "labels": [label],
            },
        })

    tasks.append({
        "data": {
            "sentence": sentence,
            "file": row.get("file", ""),
            "specialty": row.get("specialty", ""),
            "section": row.get("section", ""),
        },
        "predictions": [
            {"model_version": "regex+scispacy_v1", "result": results}
        ],
    })

print(f"Built {len(tasks)} pre-annotated tasks. Skipped {skipped} due to errors.")

Loaded 700 sentences to pre-annotate.
Built 700 pre-annotated tasks. Skipped 0 due to errors.


In [7]:
import os
os.makedirs(os.path.dirname(OUTPUT_JSON), exist_ok=True)

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)

print(f"Saved -> {OUTPUT_JSON}")

Saved -> ../data/annotated/preannotated_tasks.json


## 6. Import into Label Studio

1. Go to your project → **Settings → Labeling Interface** — confirm your
   labeling config still has the 6 tags (DRUG, DOSAGE, FREQUENCY, DURATION,
   SYMPTOM, DIAGNOSIS) and `value="$sentence"`.
2. Go to **Import** (same button you used for the CSV) and upload
   `preannotated_tasks.json` instead.
3. Open a task — spans should already be highlighted and tagged.
4. **Review each task**: fix any wrong/missed spans, manually add any
   `SYMPTOM` tags (not auto-tagged), then click **Submit**.

This turns "draw ~57 spans from scratch" into "scan ~57 spans and fix the
handful that are wrong" — much faster across 700 sentences.
